# Two-stage training-noise sweep

This notebook tests the primary connected coefficient-to-mask model at sigma=0.001, 0.005, and 0.01. Noise is added only to training gradients; validation and test gradients remain clean, matching the confirmed project protocol. Predicted-coefficient IoU is the official deployed metric.

In [ ]:
from pathlib import Path
import json
import sys
import gc
import pandas as pd
import torch

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from config import (Stage1ModelConfig, Stage2ModelConfig, StageTrainingConfig,
                    TwoStageRunConfig, TwoStageStackConfig)
from final_models.two_stage import run_two_stage

N = 10
SIGMAS = (0.001, 0.005, 0.01)
SEED = 42
TRAINING_SAMPLES = 20_000
VALIDATION_SAMPLES = 2_000
TEST_SAMPLES = 1_000
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_ROOT = ROOT / 'outputs' / 'final_two_stage_sigma_sweep'
NOTES_ROOT = ROOT / 'docs' / 'experiments' / 'sigma01_results'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
NOTES_ROOT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(2)
print('Device:', DEVICE)
print('Training noise only; validation/test remain clean')

In [ ]:
def make_config(sigma):
    stage1 = Stage1ModelConfig(
        hidden_layer_sizes=(256, 512, 256), dropout_rates=(0.2, 0.2, 0.2),
        training=StageTrainingConfig(
            epochs=200, batch_size=64, learning_rate=0.0005, validation_frequency=20,
            verbose=False, early_stopping_patience=20, min_epochs=40,
            min_improvement=0.001, lr_drop_factor=0.5, lr_drop_period=60,
            weight_decay=0.0001, gradient_clip_norm=1.0, loss_type='mse',
        ),
    )
    stage2 = Stage2ModelConfig(
        hidden_layer_sizes=(512, 1024), dropout_rates=(0.1, 0.1),
        model_type='coord_conv_decoder', latent_grid_size=16, latent_channels=160,
        decoder_channels=(160, 128, 96, 64, 32), use_rectangle_edge_weighting=True,
        use_foreground_pos_weight=False, rectangle_edge_weight=4.0, rectangle_edge_width=3,
        edge_weight_mode='rectangle', annulus_edge_weight=1.0, annulus_edge_width=3,
        training=StageTrainingConfig(
            epochs=170, batch_size=96, learning_rate=0.0005, validation_frequency=60,
            verbose=False, early_stopping_patience=25, min_epochs=50,
            min_improvement=0.001, lr_drop_factor=0.5, lr_drop_period=80,
            weight_decay=0.00025, gradient_clip_norm=0.8, loss_type='bce_dice',
            dice_loss_weight=1.0, dice_smooth=1.0,
        ),
    )
    return TwoStageRunConfig(
        N=N, training_samples=TRAINING_SAMPLES, validation_samples=VALIDATION_SAMPLES,
        test_samples=TEST_SAMPLES, noise_sigma=sigma, noise_mode='absolute', seed=SEED,
        training_noise_replicas=2, use_validation_threshold_sweep=True,
        training_shape_weights=(('rectangle', 0.25), ('two_circles', 0.45),
                                ('annulus', 0.10), ('ellipse', 0.15), ('circle', 0.05)),
        stage2_predicted_coefficient_augmentation_copies=2,
        stage2_predicted_coefficient_noise_scale=0.5,
        stage2_include_gradient_features=False,
        model=TwoStageStackConfig(stage1=stage1, stage2=stage2),
        output_dir=OUTPUT_ROOT / f'sigma_{sigma:g}',
    )

def shape_rows(summary, sigma):
    rows = []
    for shape, metrics in summary['metrics_by_shape']['stage2_test'].items():
        rows.append({'sigma': sigma, 'shape': shape, **metrics})
    return rows

In [ ]:
results = []
shape_results = []
for sigma in SIGMAS:
    print(f'\n=== TWO-STAGE sigma={sigma:g} ===', flush=True)
    config = make_config(sigma)
    summary = run_two_stage(config, device=DEVICE)
    row = {
        'sigma': sigma, 'N': N, 'base_training_samples': TRAINING_SAMPLES,
        'actual_training_rows': TRAINING_SAMPLES * 2,
        'predicted_test_iou': summary['metrics']['stage2_test']['mean_iou'],
        'true_coefficient_test_iou': summary['diagnostics']['stage2_with_true_coefficients']['test']['mean_iou'],
        'predicted_fixed_iou': summary['metrics']['stage2_fixed']['mean_iou'],
        'true_coefficient_fixed_iou': summary['diagnostics']['stage2_with_true_coefficients']['fixed']['mean_iou'],
        'validation_iou': summary['threshold_summary']['validation_metrics']['mean_iou'],
        'threshold': summary['threshold_summary']['selected_threshold'],
    }
    results.append(row)
    shape_results.extend(shape_rows(summary, sigma))
    print(json.dumps(row, indent=2), flush=True)
    print(pd.DataFrame(shape_rows(summary, sigma)).to_string(index=False), flush=True)
    del summary, config
    gc.collect()

comparison = pd.DataFrame(results).sort_values('sigma')
by_shape = pd.DataFrame(shape_results).sort_values(['sigma', 'shape'])
comparison.to_csv(NOTES_ROOT / 'two_stage_noise_sweep.csv', index=False)
by_shape.to_csv(NOTES_ROOT / 'two_stage_noise_by_shape.csv', index=False)
(NOTES_ROOT / 'two_stage_noise_sweep.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
print('\n=== TWO-STAGE SWEEP RESULTS ===')
print(comparison.to_string(index=False))
comparison